# Auditing AI4I 2020 (UCI #601)

A predictive-maintenance benchmark whose failure label is generated by documented threshold rules, with the per-mode flags shipped as columns. The audit recovers the rule system and, in its residuals, isolates the dataset's nine known generator-bug rows.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'datasets').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from synthaudit import Audit
df = pd.read_csv(ROOT / 'datasets' / 'ai4i2020.csv')
audit = Audit(df, target='Machine failure', name='ai4i2020')
results = audit.run()

[synthaudit] auditing 'ai4i2020' (10,000 rows x 14 cols, target=Machine failure)


[synthaudit] profile: 12 numeric, 7 categorical, 0 duplicate rows


[synthaudit] identity mining: 0 exact, 0 near identities, 0 FDs, 4 rule-derived labels


[synthaudit] determinism sweep: 0 deterministic, 2 near-deterministic columns


[synthaudit] causal scan: 9 edges on stochastic core (3 columns excluded)


[synthaudit] leakage audit: 2 critical, 0 high findings


[synthaudit] BTI = 0.227 (grade F) pillars={'L': 0.009, 'F': 0.6154, 'H': 1.0, 'R': 0.9722, 'I': 1.0}


[synthaudit] done in 14.07s


In [2]:
rule = [r for r in results['identity']['rule_derived_labels']
        if r['target'] == 'Machine failure'][0]
print('fidelity:', rule['fidelity'], '| violating rows:', rule['violating_rows_in_sample'])
print(rule['rules'][:500])

fidelity: 0.9991 | violating rows: 9
|--- HDF <= 0.50
|   |--- PWF <= 0.50
|   |   |--- OSF <= 0.50
|   |   |   |--- TWF <= 0.50
|   |   |   |   |--- Rotational speed [rpm] <= 1439.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- Rotational speed [rpm] >  1439.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- TWF >  0.50
|   |   |   |   |--- class: 1
|   |   |--- OSF >  0.50
|   |   |   |--- class: 1
|   |--- PWF >  0.50
|   |   |--- class: 1
|--- HDF >  0.50
|   |--- class: 1



Nine rows violate the recovered rule. Cross-check them directly: rows where the shipped label says failure while every failure-mode flag is zero.

In [3]:
flags = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
bug = df[(df['Machine failure'] == 1) & (df[flags].sum(axis=1) == 0)]
print(len(bug), 'label-contradicts-flags rows')
bug[['UDI', 'Machine failure'] + flags].head(9)

9 label-contradicts-flags rows


,UDI,Machine failure,TWF,HDF,PWF,OSF,RNF
1437,1438,1,0,0,0,0,0
2749,2750,1,0,0,0,0,0
4044,4045,1,0,0,0,0,0
4684,4685,1,0,0,0,0,0
5536,5537,1,0,0,0,0,0
5941,5942,1,0,0,0,0,0
6478,6479,1,0,0,0,0,0
8506,8507,1,0,0,0,0,0
9015,9016,1,0,0,0,0,0


In [4]:
print('grade:', results['scoring']['grade'])
print('label components:', [c for c, r in results['taxonomy']['roles'].items()
                            if r['role'] == 'label_component'])
print('honest score vs baseline:',
      results['scoring']['evidence']['honest_model_score'],
      results['scoring']['evidence']['honest_baseline'])

grade: F
label components: ['Rotational speed [rpm]', 'TWF', 'HDF', 'PWF', 'OSF']
honest score vs baseline: 0.9779 0.9694


The 99.9%-style accuracies reported on this dataset restate the generating rules. On the honest view, skill over the base rate is modest, and that is the real benchmark.